## Imports

In [5]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import sys
import os
import datetime
import math
import scipy

from openradar.mmwave import dsp
from openradar.mmwave.dataloader.adc import DCA1000
from openradar.mmwave.dsp.range_processing import range_processing
from openradar.mmwave.dsp.doppler_processing import doppler_processing
from openradar.mmwave.dsp.utils import Window

## FFT Analysis
This section analyses and plots Range and Doppler FFTs

### Range FFT with GIF plot

In [ ]:
DATA_PATH= r"../data\Rock_vibration\adc_data_2026-06-12_11-55-04_rock_layer_staticto100.npy"
DATA_PATH= DATA_PATH.replace("\\","/")

FILENAME=DATA_PATH.split("/")[-1].split(".")[0][29:]

save_dir = f"../Simulations/Radar/{FILENAME}"
os.makedirs(save_dir, exist_ok=True)

FILENAME=DATA_PATH.split("/")[-1].split(".")[0][29:]
print(FILENAME)
numFrames = 300
numADCSamples = 256
numTxAntennas = 3
numRxAntennas = 4
numLoopsPerFrame = 182
numChirpsPerFrame = numTxAntennas * numLoopsPerFrame

../data/Rock_vibration/adc_data_2026-06-12_11-55-04_rock_layer_staticto100.npy


In [ ]:
adc_data = np.load(DATA_PATH)
print("Raw data shape : " , adc_data.shape)

adc_data = np.apply_along_axis(DCA1000.organize, 1, adc_data,num_chirps=numChirpsPerFrame,num_rx=numRxAntennas, num_samples=numADCSamples)
print("Reshaped data shape: ", adc_data.shape)

print("Generating radar cubes...")
radar_cubes = np.array([range_processing(frame) for frame in adc_data])

print("Generating GIF...")
fig, ax = plt.subplots(figsize=(14, 5))
bins = np.arange(256)
global_max = np.max(np.abs(radar_cubes[:, 0, 0, :]))

magnitude_data_initial = np.abs(radar_cubes[0][0, 0, :])
line, = ax.plot(bins, magnitude_data_initial)

ax.set_xlabel("Range Bins")
ax.set_ylabel("Magnitude")
ax.grid(True)

ax.set_ylim(0, global_max * 1.1 + 1e-6)
def update(frame):
    magnitude_data = np.abs(radar_cubes[frame][0, 0, :])
    line.set_ydata(magnitude_data)
    ax.set_title(f"Range Profile (Frame {frame})")
    return [line]

ani = FuncAnimation(
    fig, 
    update, 
    frames=len(radar_cubes), 
    interval=50, 
    blit=False
)

target_folder = os.path.join(save_dir, "FFTs")
os.makedirs(target_folder, exist_ok=True)
ani.save(f"{save_dir}/FFTs/Range.gif", writer="pillow")

### Doppler FFT GIF Plot

In [ ]:
det_matrices = []
aoa_inputs = []

for cube in radar_cubes:
    det_matrix, aoa_input = doppler_processing(
        cube,
        num_tx_antennas=3, 
        clutter_removal_enabled=False,
        interleaved=True, 
        window_type_2d=Window.HANNING,
        accumulate=True     
    )
    det_matrices.append(det_matrix)
    aoa_inputs.append(aoa_input)

det_matrices = np.array(det_matrices)
aoa_inputs = np.array(aoa_inputs)

print("Generating doppler plot...")
rd_map = det_matrices[50]
rd_map_shifted = np.fft.fftshift(rd_map, axes=1)

global_vmin = np.min(det_matrices)
global_vmax = np.max(det_matrices)

fig, ax = plt.subplots(figsize=(10, 8))
initial_map = np.fft.fftshift(det_matrices[0], axes=1)

num_range_bins, num_doppler_bins = rd_map_shifted.shape
extent = [-num_doppler_bins // 2, num_doppler_bins // 2, 0, num_range_bins]

im = ax.imshow(
    initial_map, 
    aspect='auto', 
    origin='lower', 
    cmap='jet', 
    extent=extent,
    vmin=global_vmin, 
    vmax=global_vmax
)

ax.set_xlabel("Doppler Bins")
ax.set_ylabel("Range Bins")
cbar = fig.colorbar(im, label="Magnitude")

def update(frame):
    rd_map_shifted = np.fft.fftshift(det_matrices[frame], axes=1)
    im.set_array(rd_map_shifted)
    ax.set_title(f"Range-Doppler Heatmap (Frame {frame})")
    return [im]

ani = FuncAnimation(
    fig, 
    update, 
    frames=len(det_matrices), 
    interval=50, 
    blit=False  
)

ani.save(f"{save_dir}/FFTs/Doppler.gif", writer="pillow")

## Phase Analysis

### Cleaned phase plotting

In [ ]:
def iterative_range_bins_detection(rangeResult, min_bin=10, max_bin=None):
    rangeResult = np.transpose(np.stack([rangeResult[0::3], rangeResult[1::3], rangeResult[2::3]], axis=1),axes=(1,2,0,3))
    range_result_absnormal_split=[]
    
    for i in range(numTxAntennas):
        for j in range(numRxAntennas):
            r_r=np.abs(rangeResult[i][j])
    
            r_r[:, :min_bin] = 0
            if max_bin is not None:
                r_r[:, max_bin:] = 0 
                
            min_val = np.min(r_r)
            max_val = np.max(r_r)
            
            if max_val == min_val: 
                r_r_normalise = np.zeros_like(r_r)
            else:
                r_r_normalise = (r_r - min_val) / (max_val - min_val) * (1000 - 0) + 0
                
            range_result_absnormal_split.append(r_r_normalise)
    
    range_abs_combined_nparray=np.zeros((numLoopsPerFrame,numADCSamples))
    for ele in range_result_absnormal_split:
        range_abs_combined_nparray+=ele
    range_abs_combined_nparray/=(numTxAntennas*numRxAntennas)
    
    range_abs_combined_nparray_collapsed=np.sum(range_abs_combined_nparray,axis=0)/numLoopsPerFrame
    peaks_min_intensity_threshold = np.argsort(range_abs_combined_nparray_collapsed)[::-1][:max_bin-min_bin]
    max_range_index=np.argmax(range_abs_combined_nparray_collapsed)
    
    return max_range_index, peaks_min_intensity_threshold, rangeResult

def get_phase(r, i):
    return np.angle(complex(r, i))

def phase_unwrapping(phase_len,phase_cur_frame):
    i=1
    new_signal_phase = phase_cur_frame
    for k,ele in enumerate(new_signal_phase):
        if k==len(new_signal_phase)-1:
            continue
        if new_signal_phase[k+1] - new_signal_phase[k] > 1.5*np.pi:
            new_signal_phase[k+1:] = new_signal_phase[k+1:] - 2*np.pi*np.ones(len(new_signal_phase[k+1:]))
        if new_signal_phase[k+1] - new_signal_phase[k] < -1.5*np.pi:
            new_signal_phase[k+1:] = new_signal_phase[k+1:] + 2*np.pi*np.ones(len(new_signal_phase[k+1:]))
    return np.array(new_signal_phase)

def solve_equation(phase_cur_frame):
    phase_diff=[]
    for soham in range (1,len(phase_cur_frame)):
        phase_diff.append(phase_cur_frame[soham]-phase_cur_frame[soham-1])
    L=100
    r0=20
    roots_of_frame=[]
    for i,val in enumerate(phase_diff):
        c=(phase_diff[i]*0.001/3.14)/(3*(Tp+Tc))
        t=3*(i+1)*(Tp+Tc)
        c1=t*t
        c2=-2*L*t
        c3=L*L-c*c*t*t
        c4=2*L*c*c*t
        c5=-r0*r0*c*c
        coefficients=[c1, c2, c3, c4, c5]
        root=min(np.abs(np.roots(coefficients)))
        roots_of_frame.append(root)
    median_root=np.median(roots_of_frame)
    final_roots=[]
    for root in roots_of_frame:
        if root >0.9*median_root and root<1.1*median_root:
            final_roots.append(root)
    return np.mean(final_roots)

def get_velocity_antennawise(range_FFT_,peak):
    phase_per_antenna=[]
    vel_peak=[]
    for k in range(0,numLoopsPerFrame):
        r = range_FFT_[k][peak].real
        i = range_FFT_[k][peak].imag
        phase=get_phase(r,i)
        phase_per_antenna.append(phase)
    phase_cur_frame=phase_unwrapping(len(phase_per_antenna),phase_per_antenna)
    cur_vel=solve_equation(phase_cur_frame)
    return cur_vel

def get_phase_antennawise(range_FFT_, peak):
    phase_per_antenna = []
    for k in range(numLoopsPerFrame):
        r = range_FFT_[k][peak].real
        i = range_FFT_[k][peak].imag
        phase_per_antenna.append(get_phase(r, i))
    
    phase_cur_frame = np.unwrap(phase_per_antenna)
    phase_cur_frame -= np.mean(phase_cur_frame)   # <-- add this
    return phase_cur_frame

def get_averaged_phase(rangeResult, target_bin):
    best_snr = -np.inf
    best_phase = None
    for tx in range(numTxAntennas):
        for rx in range(numRxAntennas):
            signal = rangeResult[tx][rx][:, target_bin]
            snr = np.mean(np.abs(signal)**2)
            if snr > best_snr:
                best_snr = snr
                best_phase = np.unwrap(np.angle(signal))
    best_phase -= np.mean(best_phase)
    return best_phase